# Step-by-Step Implementation Guide (4-Hour Plan)

This guide breaks down the implementation into 5 sequential steps so you can build and verify the system piece by piece.

---

## Step 1: Project Scaffolding & Docker (30 minutes)

**Goal:** Create the folders, initialize projects, and set up Docker so you can test easily.

1. **Create the directories:**
   ```bash
   mkdir frontend api analysis
   ```
2. **Initialize Node API:**
   ```bash
   cd api
   npm init -y
   npm install express mongoose cors axios
   ```
3. **Initialize Python Service:**
   ```bash
   cd ../analysis
   # Create requirements.txt
   echo "fastapi\nuvicorn\nchromadb\nsentence-transformers\npydantic" > requirements.txt
   ```
4. **Create `docker-compose.yml` in the root:**
   Write a basic compose file defining 4 services: `mongo` (image: mongo:7), `api` (build: ./api, ports: 3001:3001), `analysis` (build: ./analysis, ports: 8000:8000), and `frontend` (build: ./frontend, ports: 5173:5173).

---

## Step 2: The Python RAG Service (45 minutes)

**Goal:** Build a standalone FastAPI app that embeds catalog items and retrieves them.

1. **Create `analysis/main.py`:**
   - Set up FastAPI app.
   - Initialize ChromaDB client (`chromadb.Client()`).
   - Create a ChromaDB collection (`client.create_collection("parts")`).
2. **Load Initial Data into Chroma:**
   - On startup, read the 100-part JSON catalog (you can share a `seed.json` with Node).
   - Use `collection.add(documents=[part_descriptions], metadatas=[part_data], ids=[part_numbers])`. ChromaDB will automatically download and use `all-MiniLM-L6-v2` to embed them!
3. **Create `POST /analyze` Endpoint:**
   - Accept JSON: `{ "text": "user request" }`.
   - Query ChromaDB: `collection.query(query_texts=[request.text], n_results=3)`.
   - **Refusal Logic**: Check the distance/score of the top result. If the distance is too high (meaning low similarity, e.g., > 1.2 depending on the distance metric), return `flagged_for_human: true` and empty matches.
   - **Draft Reply**: Construct a deterministic string: `"Based on your request, I recommend {part1} which costs {price}."`

---

## Step 3: The Node.js Express API (60 minutes)

**Goal:** Build the state machine, validation, and database layer.

1. **Create `api/server.js`:**
   - Connect to MongoDB using Mongoose.
   - Define Mongoose Models: `Part` (catalog) and `TriageRequest` (text, status, matched_parts, draft_reply).
2. **Seed Data:**
   - On startup, check `if (await Part.countDocuments() === 0)` and insert 100 parts from `seed.json`.
3. **The Config Endpoint:**
   - `GET /api/config`: Return a JSON object with allowed statuses (`['PENDING', 'ANALYZED', 'HUMAN_REVIEW_NEEDED', 'APPROVED', 'REJECTED']`) and categories.
4. **The Core `POST /api/requests`:**
   - **Validation**: Check if `text` is empty or > 2000 chars. Return `400`.
   - **Deduplication**: `await TriageRequest.findOne({ text, createdAt: { $gt: Date.now() - 30000 } })`. Return `409` if found.
   - **Save**: Create new `TriageRequest` with status `PENDING`.
   - **Fire & Forget**: Define an async function that calls `axios.post('http://analysis:8000/analyze', { text })`. Once it resolves, update the `TriageRequest` in Mongo to `ANALYZED` (or `HUMAN_REVIEW_NEEDED`) and save the matched parts.
   - **Respond immediately** with `201 Created` (do not await the Python call).
5. **Additional Endpoints:**
   - `GET /api/requests`: Return all requests (sorted by newest).
   - `PATCH /api/requests/:id`: Update status (e.g., to `APPROVED`). Ensure you validate that the transition is allowed!

---

## Step 4: React Frontend (60 minutes)

**Goal:** Build the UI that dynamically renders based on API config.

1. **Initialize React:**
   ```bash
   npm create vite@latest frontend -- --template react
   cd frontend && npm install
   ```
2. **Fetch Config on Mount:**
   - In `App.jsx`, use `useEffect` to fetch `GET /api/config`. Store it in Context or state.
3. **Request List Component:**
   - Fetch `GET /api/requests` inside a `setInterval` every 3 seconds to get live updates.
   - Display them in a sidebar or top list. Map the status to colors provided by the config.
4. **Request Detail Component:**
   - When a request is clicked, show its text, the `draft_reply`, and cards for the `matched_parts`.
   - Add buttons for "Approve" and "Reject". When clicked, send `PATCH /api/requests/:id` and update the local state.
5. **Styling:**
   - Use pure CSS (`index.css`). Use Flexbox for layout (side-by-side on desktop, stacked on mobile). No Tailwind or UI kits.

---

## Step 5: Testing & Scenarios (45 minutes)

**Goal:** Ensure all 6 scenarios in the instructions work perfectly.

1. **Scenario 1 (Analysis down):** Add a `setInterval` in Node.js that runs every 1 minute. It finds all requests where `status === 'PENDING'` and `createdAt < Date.now() - 60000`, and re-triggers the Python HTTP call.
2. **Test out-of-catalog:** Type "10kW motor". Ensure Python flags it and React shows the human review badge.
3. **Write tests:** If you have time, write a simple `test.js` script in Node that makes a double-click POST request simultaneously to prove deduplication works.

**Total Time:** ~4 hours. Keep your code contained to a few files to make debugging and explanation extremely easy during the live session!
